In [12]:
import numpy as np
import pandas as pd
import rasterio as rio
from scipy.ndimage import distance_transform_edt
import random
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm.autonotebook import tqdm
from rasterio.transform import Affine
import xgboost as xgb
from xgboost import XGBClassifier
import os, re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.feature_selection import RFECV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

from scipy.spatial.distance import pdist, squareform
import joblib

In [13]:
country_pseudo_values = pd.read_csv('../data/country-pseudo-values.csv')
region_mask_          = rio.open('../results/region-masks/region-mask-1km.tif')
country_raster_       = rio.open('../results/country-rasters/country-raster-eck4-1km.tif')

In [14]:
region_mask           = region_mask_.read(1)
country_raster        = country_raster_.read(1)

In [ ]:
feature_list_1km = dict({
    'elevation': '../results/features/elevation-1km.tif',
    'slope': '../results/features/slope-1km.tif', 
    'population': '../results/population-rasters/population-2020-1km.tif',
    'accessibility': '../results/features/accessibility-1km.tif',
    'D2water': '../results/features/distance-to-water-1km.tif', 
    'forest': '../results/features/forest/forest-2020-1km.tif',
    'cropland': '../results/features/cropland-2019-1km.tif', 
    'water': '../results/mining-area/global-water-area-1km.tif', 
    'su': '../results/features/glim-rasters/su-1km.tif',
    'vb': '../results/features/glim-rasters/vb-1km.tif',
    'ss': '../results/features/glim-rasters/ss-1km.tif',
    'pb': '../results/features/glim-rasters/pb-1km.tif',
    'sm': '../results/features/glim-rasters/sm-1km.tif', 
    'sc': '../results/features/glim-rasters/sc-1km.tif', 
    'va': '../results/features/glim-rasters/va-1km.tif',
    'mt': '../results/features/glim-rasters/mt-1km.tif',
    'pa': '../results/features/glim-rasters/pa-1km.tif',
    'vi': '../results/features/glim-rasters/vi-1km.tif',
    'wb': '../results/features/glim-rasters/wb-1km.tif',
    'py': '../results/features/glim-rasters/py-1km.tif',
    'pi': '../results/features/glim-rasters/pi-1km.tif',
    'ev': '../results/features/glim-rasters/ev-1km.tif',
    'nd': '../results/features/glim-rasters/nd-1km.tif', 
    'ig': '../results/features/glim-rasters/ig-1km.tif', 
    'PA_Ia': '../results/protected-area-rasters/protected-area-Ia-mask.tif',
    'PA_Ib': '../results/protected-area-rasters/protected-area-Ib-mask.tif',
    'PA_II': '../results/protected-area-rasters/protected-area-II-mask.tif',
    'PA_III': '../results/protected-area-rasters/protected-area-III-mask.tif',
    'PA_IV': '../results/protected-area-rasters/protected-area-IV-mask.tif',
    'PA_V': '../results/protected-area-rasters/protected-area-V-mask.tif',
    'PA_VI': '../results/protected-area-rasters/protected-area-VI-mask.tif',
    'urban': '../results/features/urban-area-2020-1km.tif',
    'bare': '../results/features/bare-land-2020-1km.tif',
    'building_stock': '../results/features/building-mineral-stock-1km.tif',
    'building_stock_around': '../results/features/building-mineral-stock-50km-around-1km.tif',
})

In [ ]:
selected_features = ['elevation', 'slope', 'population', 'accessibility', 'D2water', 'forest', 'cropland', 'water', 'su', 'vb', 'ss', 'pb', 'sm', 'sc', 'va', 'mt', 'pa', 'vi', 'wb', 'py', 'pi', 'ev', 'nd', 'ig', 'PA_Ia', 'PA_Ib', 'PA_II', 'PA_III', 'PA_IV', 'PA_V', 'PA_VI', 'urban', 'bare', 'building_stock']
print(len(selected_features))

34


In [21]:
region_code = dict({
    'Middle East': 1, 
    'Latin America and the Caribbean': 2,
    'North Africa': 3, 
    'West Africa': 4,
    'East Africa': 5,
    'Central Africa': 6,
    'Southern Africa': 7,
    'Europe': 8, 
    'South-eastern Asia': 9, 
    'Oceania': 10,
    'Southern Asia': 11, 
    'Eastern Asia': 12, 
    'Central and Northern Asia': 13, 
    'Northern America': 14,
    'China': 15
})

In [22]:
pred_global = np.zeros_like(region_mask, dtype=np.float32)

In [ ]:
for i in tqdm(range(1, 256)):
    # print(i)
    if i in [161, 213]:
        continue
    else:
        the_region = country_pseudo_values[country_pseudo_values['value'] == i]['region'].values[0]
        region_id  = region_code[the_region]

    
    model = joblib.load(f'../results/trained-models/xgb-region-{region_id}.pkl')

    country_index = np.where(country_raster == i)

    if len(country_index[0]) == 0:
        # print(f"[Skip] Country {i} has no pixels in raster.")
        continue

    i1, i2 = np.min(country_index[0]), np.max(country_index[0])
    j1, j2 = np.min(country_index[1]), np.max(country_index[1])
    a_window = ((i1, i2), (j1, j2))

    feature_arrays = []
    for feature in selected_features:
        path = feature_list_1km[feature]
        with rio.open(path) as src:
            data = src.read(1, window=a_window).astype(np.float32)
            feature_arrays.append(data.flatten())
    
    X_country = np.stack(feature_arrays, axis=1)
    mask_valid = ~np.isnan(X_country).any(axis=1)
    X_valid = X_country[mask_valid]

    if X_valid.shape[0] == 0:
        continue

    # 预测概率
    p_valid = model.predict_proba(X_valid)[:, 1]

    # 获取窗口的二维形状（从刚读的任意特征 data 得到）
    window_shape = feature_arrays[0].reshape(i2 - i1, j2 - j1).shape

    # 构造完整预测数组并恢复形状
    pred = np.full(X_country.shape[0], np.nan, dtype=np.float32)
    pred[mask_valid] = p_valid
    pred = pred.reshape(window_shape)  # 恢复为窗口形状 (rows, cols)

    # 回填到全局栅格
    submask = (country_raster[i1:i2, j1:j2] == i)
    pred_global[i1:i2, j1:j2][submask] = pred[submask]


In [ ]:
pred_scaled = np.nan_to_num(pred_global, nan=0.0)   # 把NaN替换为0
pred_scaled = np.clip(pred_scaled, 0, 1)            # 限定在0~1之间
pred_scaled = (pred_scaled * 10000).astype(np.uint16)

In [ ]:
with rio.open(
    r'../results/mining-suitability/mining-suitability-1km.tif', 
    'w', 
    driver='GTiff', 
    height=pred_scaled.shape[0],
    width=pred_scaled.shape[1],
    count=1, 
    dtype=pred_scaled.dtype, 
    crs=country_raster_.crs, 
    transform=country_raster_.transform, 
    compress='lzw'
    ) as dst:
    dst.write(pred_scaled, 1)